# adaptive-intelligence v4: Complete Demo
## Context Engineering + MCP Tools + Agentic Workflow + Persistent Memory

### What's new in v4

| Feature | What it does |
|---------|-------------|
| Context Engineering | Optimizes entire context window — not just chunks, but memory, history, tool results, prompt |
| Tool Registry | Connect external tools (MCP servers, APIs, Python functions). RL learns which to call. |
| Agentic Mode | Multi-round retrieval. System retrieves, evaluates, refines, retrieves again. |
| Persistent Memory | Remembers across sessions. Learned patterns survive restart. |
| Incremental Learning | Add new documents anytime. System continues from current state. |

**Before running:** Runtime > Change runtime type > T4 GPU

---
[PyPI](https://pypi.org/project/adaptive-intelligence/) | [GitHub](https://github.com/VK-Ant/adaptive-intelligence) | [Paper](https://www.researchgate.net/publication/405076088)  
Author: Venkatkumar Rajan | @VK_Venkatkumar


## 1. Install Necessary Library (run once, restart runtime, then skip)

In [1]:
!pip uninstall torchvision torchaudio -y -q
!pip install torchvision torchaudio -q
print('Now: Runtime > Restart runtime, then run from cell 2')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5

## 2. Install

In [1]:
%%capture
!pip install adaptive-intelligence chromadb transformers accelerate -q


## 3. Load Model

In [2]:
import torch, time, os, json, logging
from transformers import AutoModelForCausalLM, AutoTokenizer
logging.getLogger('adaptive_intelligence.evaluation').setLevel(logging.ERROR)

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'Loading {MODEL}...')
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='auto')
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f'Ready in {time.time()-t0:.0f}s | GPU: {torch.cuda.get_device_name(0)}')


Loading Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Ready in 57s | GPU: Tesla T4


## 4. Create Documents

Two scenarios: **Financial** (5 docs) and **Healthcare** (3 docs).
We'll show how the system handles both domains with different retrieval strategies.


In [3]:
# Financial documents
FIN_DOCS = {
    "q3_report.txt": "NovaTech Q3 2025: Revenue $847M (+12.3% YoY). Product $612M (+15.1%), services $235M (+5.8%). Operating income $127M, margin 15.0%. EBITDA $168M, margin 19.8%. Cash $1.2B. Q4 guidance: $870-890M, 15.5-16.0% margin.",
    "q2_report.txt": "NovaTech Q2 2025: Revenue $798M (+9.7% YoY). Margin 13.8%. Meridian Semiconductors 3-week delay disrupted APAC. R&D spending $95M for AI analytics.",
    "risks.txt": "Risk 2025: Supply Chain (HIGH) - 65% Meridian dependency. Pacific Chip Alliance secondary supplier, target 45% by Q2 2026. Cybersecurity (MEDIUM) - CyberShield Partners, $12M zero-trust.",
    "company.txt": "CEO Sarah Chen. CloudBridge Solutions (100%, $340M), DataStream Analytics (75%, JV Apex Capital). Competitors: AscentTech, Vertex Digital.",
    "operations.txt": "Operations H1 2025: Yield 97.2%. NovaStar Edge: 340 customers. Headcount 12,400. Carbon -22%. Renewable 68%.",
}

# Healthcare documents
MED_DOCS = {
    "patient_guidelines.txt": "Treatment Protocol 2025: Type 2 Diabetes management. First-line: Metformin 500mg twice daily. If HbA1c > 7.5% after 3 months, add SGLT2 inhibitor (Empagliflozin 10mg). Monitor kidney function (eGFR) every 6 months. Contraindicated if eGFR < 30. Side effects: GI disturbance (Metformin), UTI risk (SGLT2).",
    "drug_interactions.txt": "Drug Interactions: Metformin + contrast dye = risk of lactic acidosis, hold 48h before procedure. SGLT2 + loop diuretics = dehydration risk, monitor volume. ACE inhibitors + potassium supplements = hyperkalemia. Warfarin + NSAIDs = increased bleeding risk.",
    "clinical_trials.txt": "Trial NCT-2025-0847: Empagliflozin cardiovascular outcomes. N=4200, 36-month follow-up. Primary endpoint: MACE reduction 23% vs placebo (p=0.003). Secondary: hospitalization for heart failure reduced 35%. Subgroup: eGFR 30-60 showed 28% MACE reduction. Adverse: genital infections 8% vs 2% placebo.",
}

for name, docs in [('Financial', FIN_DOCS), ('Healthcare', MED_DOCS)]:
    d = f'/content/{name.lower()}_docs'
    os.makedirs(d, exist_ok=True)
    for n, c in docs.items():
        with open(os.path.join(d, n), 'w') as f: f.write(c)
    print(f'{name}: {len(docs)} documents')


Financial: 5 documents
Healthcare: 3 documents


---
## DEMO 1: Basic Usage + Incremental Learning

Ingest documents. Ask questions. Then add MORE documents later.
The system continues from its current learned state — no restart needed.
RL policy, graph, memory all preserved when you add new docs.


In [4]:
from adaptive_intelligence import AdaptiveAI

engine = AdaptiveAI(
    llm_backend='huggingface', llm_model=MODEL,
    domain='financial', storage_dir='/content/state_v4',
    log_level='ERROR',
)

# Initial ingestion
stats = engine.ingest('/content/financial_docs')
print(f'Initial: {stats.total_chunks} chunks, {engine.graph.node_count} graph nodes')

# Ask a question
r = engine.ask('What is Q3 revenue?')
print(f'\nAnswer: {r.answer[:300]}')
print(f'Strategy: {r.retrieval_strategy}')


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 32.1MiB/s]


Initial: 5 chunks, 31 graph nodes


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Answer: According to NovaTech's report in "q3_report.txt", their Q3 (third quarter) reported revenue was $84 million ($847 M +$798 - $12 = -$1 billion).
This figure represents both product and service revenues combined as stated directly within that source text without any subtraction or adjustment noted el
Strategy: table_first + depth=2


In [5]:
# INCREMENTAL: Add healthcare docs LATER — system continues learning
rl_arms_before = len(engine.rl._arms)
graph_before = engine.graph.node_count

stats2 = engine.ingest('/content/healthcare_docs')
print(f'Added: {stats2.total_chunks} chunks')
print(f'Graph: {graph_before} nodes -> {engine.graph.node_count} nodes')
print(f'RL arms: {rl_arms_before} -> {len(engine.rl._arms)} (preserved + new)')

# Now ask healthcare question — system handles both domains
r = engine.ask('What is the first-line treatment for Type 2 Diabetes?')
print(f'\nAnswer: {r.answer[:300]}')
print(f'Strategy: {r.retrieval_strategy}')
print('\nRL policy preserved. No restart. New docs integrated.')


Added: 3 chunks
Graph: 31 nodes -> 52 nodes
RL arms: 1 -> 1 (preserved + new)

Answer: The first line of therapy recommended by Treatment Protocol 2
Strategy: keyword_only + depth=3

RL policy preserved. No restart. New docs integrated.


---
## DEMO 2: RL Learns Different Strategies Per Domain

Watch how the system picks different strategies for financial vs healthcare queries.
Same engine, same RL policy, different domains detected automatically.


In [6]:
queries = [
    ('What is EBITDA margin?', 'financial'),
    ('What is Metformin dosage?', 'healthcare'),
    ('How is Meridian connected to risk?', 'financial-relational'),
    ('What are Empagliflozin drug interactions?', 'healthcare-relational'),
    ('Compare Q2 vs Q3 revenue', 'financial-comparative'),
    ('Empagliflozin trial results vs placebo?', 'healthcare-comparative'),
]

print(f'{"Strategy":<28} {"Graph":<6} {"Conf":<6} {"Type":<25} Query')
print('-' * 100)
for q, qtype in queries:
    r = engine.ask(q)
    g = 'YES' if r.retrieval_info and r.retrieval_info.graph_activated else ''
    print(f'{r.retrieval_strategy:<28} {g:<6} {r.confidence:.0%}    {qtype:<25} {q}')


Strategy                     Graph  Conf   Type                      Query
----------------------------------------------------------------------------------------------------
table_first + depth=2               35%    financial                 What is EBITDA margin?
keyword_only + depth=2              45%    healthcare                What is Metformin dosage?
graph_hybrid + graph(2-hop) + depth=6 YES    30%    financial-relational      How is Meridian connected to risk?
keyword_only + depth=3              42%    healthcare-relational     What are Empagliflozin drug interactions?
hybrid + depth=7                    35%    financial-comparative     Compare Q2 vs Q3 revenue
hybrid + depth=5                    43%    healthcare-comparative    Empagliflozin trial results vs placebo?


---
## DEMO 3: Persistent Memory

The system remembers across queries. It stores:
- Which strategies worked for which query types (patterns)
- Session context for follow-up queries
- User-stored facts

Memory persists to disk. If you restart the engine, it loads from saved state.


In [7]:
# Store facts in memory
engine.remember('user_role', 'financial analyst', category='preference')
engine.remember('focus_area', 'supply chain risk', category='preference')
engine.remember('Q3_revenue', '$847M', category='fact')

# Recall
print('Stored facts:')
print(f'  Role: {engine.recall("user_role")}')
print(f'  Focus: {engine.recall("focus_area")}')
print(f'  Q3 Revenue: {engine.recall("Q3_revenue")}')

# Search memory
print(f'\nMemory search for "supply chain":')
results = engine.search_memory('supply chain risk')
for entry in results:
    print(f'  {entry.key}: {entry.value}')

# Show learned patterns
if engine._persistent_memory:
    stats = engine._persistent_memory.get_stats()
    print(f'\nMemory stats: {stats}')


Stored facts:
  Role: financial analyst
  Focus: supply chain risk
  Q3 Revenue: $847M

Memory search for "supply chain":
  focus_area: supply chain risk

Memory stats: {'total_entries': 10, 'categories': {'pattern': 7, 'preference': 2, 'fact': 1}, 'session_context_length': 8}


---
## DEMO 4: Context Engineering

v4 doesn't just retrieve chunks. It assembles the ENTIRE context window:
- System prompt (domain-aware persona)
- Relevant memory entries
- Session history (previous Q&A)
- Retrieved chunks (RL-selected)
- Tool results (if any tools connected)

Each component has a token budget. The system optimizes what goes in.


In [8]:
# Show what context engineering builds
from adaptive_intelligence.context import ContextEngineer

ce = engine._context_engineer
print('Context Engineering config:')
print(f'  Token budget: {ce.budget.total}')
print(f'  System prompt: {ce.budget.system_prompt} tokens')
print(f'  Memory: {ce.budget.memory} tokens')
print(f'  History: {ce.budget.history} tokens')
print(f'  Chunks: {ce.budget.chunks} tokens')
print(f'  Tool results: {ce.budget.tool_results} tokens')

print(f'\nAvailable personas: {list(ce.PERSONAS.keys())}')

# Ask a question — context engineering assembles everything
r = engine.ask('What supply chain risks should I monitor?')
print(f'\nAnswer: {r.answer[:300]}')
print(f'Strategy: {r.retrieval_strategy}')
print('\nContext included: system prompt + memory + history + chunks')


Context Engineering config:
  Token budget: 4096
  System prompt: 200 tokens
  Memory: 300 tokens
  History: 400 tokens
  Chunks: 2500 tokens
  Tool results: 500 tokens

Available personas: ['general', 'financial', 'legal', 'healthcare', 'technical']

Answer: Based on your request for monitoring specific supply chain risks related to Empaglifiolozin's use within healthcare settings or pharmaceutical companies like yours, several key areas need attention due to their high-risk status according to recent studies and industry practices:

### High Risk Categ
Strategy: graph_hybrid + graph(2-hop) + depth=5

Context included: system prompt + memory + history + chunks


---
## DEMO 5: Tool Registry + Cost Optimization

Connect external tools. The RL learns which tools to call per query.
Tools that don't help get skipped — saving API costs.

Here we register Python functions as tools (no external server needed).


In [9]:
# Define custom tools as Python functions
def financial_calculator(query, **kwargs):
    """Simple financial calculation tool."""
    q = query.lower()
    if 'growth' in q and '798' in q and '847' in q:
        growth = (847 - 798) / 798 * 100
        return f'Revenue growth Q2 to Q3: {growth:.1f}%'
    if 'margin' in q:
        return 'Operating margin: 15.0% (Q3) vs 13.8% (Q2) = +1.2pp improvement'
    return f'Financial calculation for: {query}'

def risk_scorer(query, **kwargs):
    """Risk scoring tool."""
    risks = {
        'supply chain': ('HIGH', 0.85, 'Meridian 65% dependency'),
        'cybersecurity': ('MEDIUM', 0.55, 'CyberShield Partners active'),
        'regulatory': ('MEDIUM', 0.50, 'EU AI Act pending'),
        'market': ('MEDIUM', 0.45, 'Share declined to 26%'),
    }
    found = []
    for risk, (level, score, detail) in risks.items():
        if risk in query.lower():
            found.append(f'{risk}: {level} (score: {score}) - {detail}')
    return '\n'.join(found) if found else 'No matching risks found'

# Register tools
engine.add_tool('financial_calc', description='financial calculations growth margin',
                function=financial_calculator)
engine.add_tool('risk_scorer', description='risk assessment scoring supply chain',
                function=risk_scorer)

print('Tools registered:')
for t in engine.list_tools():
    print(f'  {t["name"]} ({t["type"]}) - {t["description"]}')


Tools registered:
  financial_calc (function) - financial calculations growth margin
  risk_scorer (function) - risk assessment scoring supply chain


In [10]:
# Query that triggers tool call
r = engine.ask('What is the supply chain risk score?')
print(f'Answer: {r.answer[:400]}')
print(f'Strategy: {r.retrieval_strategy}')

# Show tool stats — which tools were called
print(f'\nTool stats:')
for t in engine.list_tools():
    print(f'  {t["name"]}: {t["calls"]} calls, success rate: {t["success_rate"]:.0%}')


Answer: Based on my previous response and additional research into drug interactions mentioned earlier, it appears that one relevant source provides detailed insights including scores such as High Risk Scores which indicate potential issues with dependencies among suppliers. The example given shows high-risk scoring at company level but not specifically mentioning individual product lines directly linked 
Strategy: graph_hybrid + graph(2-hop) + depth=5

Tool stats:
  financial_calc: 0 calls, success rate: 0%
  risk_scorer: 1 calls, success rate: 100%


### Cost optimization

The RL learns which tools to skip. If a tool doesn't improve answer quality,
the system stops calling it. Fewer tool calls = lower API costs.

Similarly, RL learns optimal retrieval depth per query:
- Factual query: depth 2 (2 chunks sent to LLM)
- Complex query: depth 8 (8 chunks sent to LLM)

Fewer chunks = fewer tokens = lower LLM cost. At 1000 queries/day with GPT-4o pricing,
sending 2 chunks instead of 10 for 60% of queries saves ~$50-100/month.


---
## DEMO 6: Agentic Workflow

Multi-round retrieval. The system:
1. Retrieves with RL-selected strategy
2. Evaluates if the answer is confident enough
3. If not: refines the query, calls tools, retrieves again
4. Repeats up to 3 rounds
5. Synthesizes final answer from accumulated context


In [11]:
r = engine.ask(
    'Analyze the complete supply chain risk: Meridian dependency, '
    'Pacific Chip mitigation timeline, and impact on Q4 guidance',
    mode='agentic'
)

print(f'Answer: {r.answer[:400]}')
print(f'\nAgent details:')
if hasattr(r, 'agent_rounds'):
    print(f'  Rounds: {r.agent_rounds}')
if hasattr(r, 'agent_steps'):
    for i, step in enumerate(r.agent_steps):
        print(f'  Step {i+1}: {step.action} - {step.query[:60]}...' if step.query else f'  Step {i+1}: {step.action}')
if hasattr(r, 'tools_called') and r.tools_called:
    print(f'  Tools called: {r.tools_called}')


Answer: Based on the given text, here's how I would analyze these aspects related to supply chain risks faced by Nova Tech Inc.: 

### Meridian Dependency 
**Score:** The **tool 'risk-scorer'** assigns this metric a *High* level due to a strong dependency rate of approximately ***65***%.

This means that Nova Tech relies heavily on supplies or components supplied primarily through Meridian Semiconductor P

Agent details:
  Rounds: 3
  Step 1: retrieve - Analyze the complete supply chain risk: Meridian dependency,...
  Step 2: tool_call
  Step 3: refine - Analyze the complete supply chain risk: Meridian dependency,...
  Step 4: retrieve - Analyze the complete supply chain risk: Meridian dependency,...
  Step 5: tool_call
  Step 6: refine - Analyze the complete supply chain risk: Meridian dependency,...
  Step 7: retrieve - Analyze the complete supply chain risk: Meridian dependency,...
  Tools called: ['risk_scorer', 'risk_scorer']


---
## DEMO 7: Dashboard

In [12]:
print(engine.dashboard())
print()
print('Memory:', engine._persistent_memory.get_stats() if engine._persistent_memory else 'disabled')
print('Tools:', engine._tool_registry.get_stats())


+---------------------------------------------------------+
|  ADAPTIVE INTELLIGENCE v2 DASHBOARD                    |
|                                                         |
|  Mode:                 Vector+Keyword                    |
|  Documents Indexed:         8                          |
|  Queries Processed:        13                          |
|  Average Accuracy:     38.9%                          |
|  Improvement Rate:     -32.1%                          |
|                                                         |
|  RL Policy:                Warmup                    |
|  Exploration Rate:     20.0%                          |
|  Arms Learned:             10                          |
|                                                         |
|  Graph Nodes:              52                          |
|  Graph Edges:             288                          |
|  Graph Success Rate:    0.0%                          |
+------------------------------------------------------

---
## DEMO 8: Works Without LLM

All v4 features work without an LLM:
- RL routing still learns
- Memory still persists
- Tools still callable
- Context engineering still assembles
- You just get document excerpts instead of synthesized text


In [13]:
no_llm = AdaptiveAI(
    llm_backend='none', vectorless=True,
    storage_dir='/content/state_nollm', log_level='ERROR',
)
no_llm.ingest('/content/financial_docs')
no_llm.add_tool('risk_scorer', description='risk scoring', function=risk_scorer)

r = no_llm.ask('Q3 revenue?')
print(f'Answer: {r.answer[:300]}')
print(f'\nNo LLM. No API. RL + graph + memory + tools still work.')


Answer: From q3_report.txt:
NovaTech Q3 2025: Revenue $847M (+12.3% YoY). Product $612M (+15.1%), services $235M (+5.8%). Operating income $127M, margin 15.0%. EBITDA $168M, margin 19.8%. Cash $1.2B. Q4 guidance: $870-890M, 15.5-16.0% margin.

From q2_report.txt:
NovaTech Q2 2025: Revenue $798M (+9.7% YoY).

No LLM. No API. RL + graph + memory + tools still work.


---
## Summary

| Demo | Feature | Key takeaway |
|------|---------|-------------|
| 1 | Incremental learning | Add docs anytime, system continues learning |
| 2 | Multi-domain RL | Different strategies for financial vs healthcare |
| 3 | Persistent memory | Remembers patterns, facts, preferences across sessions |
| 4 | Context engineering | Optimizes entire context window, not just chunks |
| 5 | Tool registry | Connect tools, RL learns which to call, skip unnecessary = save cost |
| 6 | Agentic workflow | Multi-round retrieval with query refinement |
| 7 | Dashboard | Full visibility |
| 8 | No LLM | Everything works without LLM |

### Cost optimization summary

| What RL optimizes | How it saves cost |
|-------------------|-------------------|
| Retrieval depth | Depth 2 for factual (not 10) = 80% fewer tokens |
| Tool selection | Skip tools that don't help = fewer API calls |
| Graph activation | Skip graph on 70% of queries = less compute |
| Context assembly | Include only relevant memory/history = fewer tokens |

---

**pip install adaptive-intelligence**

[PyPI](https://pypi.org/project/adaptive-intelligence/) | [GitHub](https://github.com/VK-Ant/adaptive-intelligence) | [Paper](https://www.researchgate.net/publication/405076088)  
Also: [llmevalkit](https://pypi.org/project/llmevalkit/) (78 metrics)  
Author: Venkatkumar Rajan | @VK_Venkatkumar
